# 🏥 CardioIA - Triagem de Risco (Classificador de Texto)

## 1. Importação de Bibliotecas e Carga de Dados

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_csv('dataset_risco.csv')

print(f"Tamanho do dataset: {df.shape[0]} amostras")
df.head()

Tamanho do dataset: 40 amostras


,frase,situacao
0,Sinto uma dor forte no peito que irradia para ...,alto risco
1,Estou com muita falta de ar mesmo em repouso.,alto risco
2,Tive um desmaio súbito e meu coração está disp...,alto risco
3,Sinto um aperto no peito e suor frio.,alto risco
4,Minha pressão subiu muito e estou com a visão ...,alto risco


## 2. Transformação de Texto (TF-IDF)

O TF-IDF atribui um peso às palavras com base em sua frequência em um documento específico em comparação com todo o conjunto de documentos. 
Ele penaliza palavras comuns (como "o", "a", "de") e destaqua palavras mais raras (como "dor", "peito", "falta").

In [7]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['frase'])
y = df['situacao']

print(f"Formato da matriz TF-IDF: {X.shape}")

Formato da matriz TF-IDF: (40, 168)


## 3. Comparação de Algoritmos

Para garantir a melhor escolha, vamos comparar diferentes algoritmos de classificação, focando em:
1. **Acurácia (Assertividade)**
2. **Recall de Alto Risco (Minimização de Falsos Negativos)**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer, recall_score

# Modelos para comparação
modelos = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42),
    'Naive Bayes': MultinomialNB(),
    'SVM (Linear)': SVC(kernel='linear', probability=True, random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42)
}

# Métrica customizada: Recall para 'alto risco'
def recall_alto(y_true, y_pred):
    return recall_score(y_true, y_pred, pos_label='alto risco')

scoring = {
    'accuracy': 'accuracy',
    'recall_alto': make_scorer(recall_alto)
}

resultados = []

for nome, clf in modelos.items():
    # [ Treino ][ Treino ][ Treino ][ Treino ][ TESTE  ] -> Score 1
    # [ Treino ][ Treino ][ Treino ][ TESTE  ][ Treino ] -> Score 2
    # [ Treino ][ Treino ][ TESTE  ][ Treino ][ Treino ] -> Score 3
    # [ Treino ][ TESTE  ][ Treino ][ Treino ][ Treino ] -> Score 4
    # [ TESTE  ][ Treino ][ Treino ][ Treino ][ Treino ] -> Score 5
    cv_res = cross_validate(clf, X, y, cv=5, scoring=scoring)
    resultados.append({
        'Algoritmo': nome,
        'Acurácia Média': cv_res['test_accuracy'].mean(),
        'Recall Alto Risco (Média)': cv_res['test_recall_alto'].mean(),
        'Risco de Falso Negativo': 1 - cv_res['test_recall_alto'].mean()
    })

df_comp = pd.DataFrame(resultados).sort_values(by='Recall Alto Risco (Média)', ascending=False)
df_comp

,Algoritmo,Acurácia Média,Recall Alto Risco (Média),Risco de Falso Negativo
3,SVM (Linear),0.825,0.95,0.05
4,Random Forest,0.850,0.90,0.10
0,Decision Tree,0.800,0.80,0.20
1,Logistic Regression,0.775,0.80,0.20
2,Naive Bayes,0.750,0.65,0.35


## 4. Análise Crítica e Governança (Escolha do Modelo SVM)

Com base na comparação anterior, o **SVM (Support Vector Machine)** foi selecionado como o classificador final para o sistema CardioIA.

### Justificativas:
1. **Falsos Negativos**: O SVM apresentou o maior **Recall para 'Alto Risco'**, minimizando a possibilidade de pacientes em estado grave serem erroneamente classificados como 'Baixo Risco'.
2. **Consistência**: O modelo demonstrou alta assertividade tanto no treino quanto nos testes via Cross-Validation.
3. **Padrões de Palavras**: O SVM linear lida bem com a natureza esparsa e de alta dimensionalidade dos vetores TF-IDF.
4. **Governança**: A priorização do Recall ao invés da Acurácia garante um sistema de atendimento "pelo erro seguro", minimizando casos onde o paciente é classificado como saúdavel mesmo tendo a doença.